# C/GMRES における初期解 $U(0)$ の求め方

|方法|特徴|
|:---|:---|
|解析的に解く|$T(0)=0$で簡単になった最適性条件から、式変形で直接求める|
|Newton-Raphson法|非線形方程式 $F(U)=0$ を反復して解く|
|Newton-GMRES法|Newton-Raphson法の各反復で出る線形方程式をGMRESで解く|
|ホモトピー法・継続法|簡単な問題の解から、ホライゾン長や制約などを徐々に変更し、目的の問題の解へ近づける|

## PMPにおける最適性条件

評価関数を最小化する最適制御問題に対して、Hamiltonian $H$ を用いると、PMPの必要条件は以下のように表される。

$$
\boxed{
\begin{aligned}
\dot{X}(\tau) &= H_\lambda(X(\tau),U(\tau),\lambda(\tau), \tau) \\
\dot{\lambda}(\tau) &= -H_X(X(\tau),U(\tau),\lambda(\tau), \tau) \\
0 &= H_U(X(\tau),U(\tau),\lambda(\tau), \tau) \\
\lambda(T) &= \Phi_X(X(T)) \\
\end{aligned}
}
$$

ここで $\tau$ は予測ホライゾン上の時刻である。

最適制御問題は、予測区間 $\tau \in [0,T]$ において、これらの条件を満たす状態軌道 $X(\tau)$, 制御入力 $U(\tau)$、随伴変数 $\lambda(\tau)$ を求める問題である。

## C/GMRES における初期解

C/GMRESでは、実時刻 $t$ に応じて予測ホライゾン長 $T(t)$ を徐々に伸ばす方法を用いることが出来る。

初期時刻において

$$
T(0) = 0
$$

と設定すると、予測区間の始点と終点が一致するため以下のようになる。

$$
X(T(0)) = X(0)
$$

また、終端条件から以下のように随伴変数が得られる。

$$
\lambda(T(0)) = \Phi_X(X(T(0))) = \Phi_X(X(0))
$$

したがって、初期時刻における停留条件は以下となる。

$$
H_U(X(0), U(0), \Phi_X(X(0)), 0) = 0
$$

ここで初期状態$X(0)$は観測値または設定値として与えられるため、未知数は$U(0)$となる。<br>
したがって、C/GMRESの初期解を求める問題は、この非線形方程式を満たす $U(0)$ を解析的または数値的に求める問題として整理できる。

なお、不等式制約をダミー入力とラグランジュ乗数を扱い場合は、以下のように制御入力 $u(0)$だけでなく、ダミー入力 $u_d(0)$ や乗数 $\rho(0)$ も含む未知変数ベクトルになる。

$$
U(0) = \begin{bmatrix} u(0) \\ u_d(0) \\ \rho(0) \end{bmatrix}
$$

## 数値的に解く

ここで以下のように残差関数$F$を定義する。

$$
F(U) \equiv H_U(X(0), U, \Phi_X(X(0)), 0) = 0
$$

$X(0)$は与えられるものであるため固定値として捉えると、求めるべき$U$により$F$の値は変化する。そこで

$$F(U)=0$$

となる $U$を求める。

### Newton法

$F(U)$ を $U$ 周りにTaylor展開を行うと、次のようになる。$J_{H_U}(U)$はヤコビアンである。

$$
F(U + \Delta U) \approx F(U) + J_{F}(U)\Delta U
$$

適当な$U^{(j)}$を考えたとき、$F(U^{(j)}) \neq 0$ である。そして、少し先の地点が $F(U^{(j)} + \Delta U) = 0$ となる、と考えると、

$$
F(U^{(j)}) + J_{F}(U^{(j)})\Delta U = 0\Rightarrow J_{F}(U^{(j)})\Delta U = -F(U^{(j)}) 
$$

よって、

$$
J_{F}(U^{(j)})\Delta U = -F(U^{(j)})
$$

を $\Delta U$ についてLU分解などで解き、

$$
U^{(j+1)} = U^{(j)} + \Delta U
$$

として更新して、$||F(U^{(j+1)})|| < \delta$ と、$F(U^{(j+1)}) \approx 0$ になるまで繰り返し計算を行う。

ここでヤコビアン $J_F(U)$ は、$F(U) = [f_1(U), f_2(U)]^T$ 、$U=[u_1, u_2]^T$ とすると

$$
J_F(U) = \begin{bmatrix}
\frac{\partial f_1}{\partial u_1} & \frac{\partial f_1}{\partial u_2} \\
\frac{\partial f_2}{\partial u_1} & \frac{\partial f_2}{\partial u_2} \\
\end{bmatrix}
$$

### Newton - GMRES法

